In [5]:
!pip install -q "mcp>=2.0" pandas groq gradio

In [32]:
%pip install mcp==2.3.0 groq==1.7.0 pandas==2.2.3


In [34]:
import pandas as pd

df = pd.read_csv("nutrition_data.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Shape: (5, 3)

Columns: ['food_name', 'calories_per_100g', 'protein_per_100g']


,food_name,calories_per_100g,protein_per_100g
0,chicken breast,165,31.0
1,salmon,208,20.0
2,egg,155,13.0
3,broccoli,34,2.8
4,tofu,76,8.0


In [35]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY").strip()

if GROQ_API_KEY:
    print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [36]:
%%writefile mcp_server.py

import logging
import sys
from pathlib import Path

import pandas as pd
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

logging.basicConfig(
    stream=sys.stderr,
    level=logging.INFO
)

logger = logging.getLogger("nutribot")

CSV_PATH = Path(__file__).parent / "nutrition_data.csv"

mcp = MCPServer("nutribot")


def load_data():
    df = pd.read_csv(CSV_PATH)

    df["food_name"] = (
        df["food_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return df


def find_food(food_name):

    if not isinstance(food_name, str) or not food_name.strip():
        raise ToolError("Please provide a valid food name.")

    key = " ".join(food_name.strip().lower().split())

    df = load_data()

    row = df[df["food_name"] == key]

    if row.empty:
        available = ", ".join(df["food_name"])

        raise ToolError(
            f"{food_name} not found. Available foods: {available}"
        )

    return row.iloc[0]


@mcp.tool()
def get_calories(food_name: str) -> str:
    """Get calories per 100 grams of a food."""
    row = find_food(food_name)

    return f"{row['calories_per_100g']:g} kcal"


@mcp.tool()
def get_protein(food_name: str) -> str:
    """Get protein per 100 grams of a food."""
    row = find_food(food_name)

    return f"{row['protein_per_100g']:g} g protein"


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting mcp_server.py


In [37]:
%%writefile mcp_client.py

import sys
from pathlib import Path
from contextlib import asynccontextmanager

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp_types import TextContent

SERVER_PATH = str(Path(__file__).parent / "mcp_server.py")

LOG_PATH = Path(__file__).parent / "mcp_server.log"


def server_params():

    return StdioServerParameters(
        command=sys.executable,
        args=[SERVER_PATH]
    )


@asynccontextmanager
async def connect():

    with open(LOG_PATH, "w", encoding="utf-8") as log_file:

        async with Client(
            stdio_client(
                server_params(),
                errlog=log_file
            )
        ) as client:

            yield client


async def discover_tools(client):

    result = await client.list_tools()

    return result.tools


async def call_tool(client, name, arguments):

    result = await client.call_tool(name, arguments)

    output = "\n".join(
        item.text
        for item in result.content
        if isinstance(item, TextContent)
    )

    return output, bool(result.is_error)

Overwriting mcp_client.py


In [38]:
async with mcp_client.connect() as client:

    print("MCP connection successful!")

    tools = await mcp_client.discover_tools(client)

    print("\nAvailable tools:")

    for tool in tools:
        print(tool.name, "-", tool.description)

MCP connection successful!

Available tools:
get_calories - Get calories per 100 grams of a food.
get_protein - Get protein per 100 grams of a food.


In [39]:
async with mcp_client.connect() as client:

    calories, error1 = await mcp_client.call_tool(
        client,
        "get_calories",
        {"food_name": "chicken breast"}
    )

    protein, error2 = await mcp_client.call_tool(
        client,
        "get_protein",
        {"food_name": "salmon"}
    )

    print("Chicken calories:", calories)
    print("Salmon protein:", protein)

    print("Errors:", error1, error2)

Chicken calories: 165 kcal
Salmon protein: 20 g protein
Errors: False False


In [40]:
%%writefile nutrition_agent.py

import json
from groq import Groq
from google.colab import userdata

import mcp_client

MODEL = "openai/gpt-oss-20b"

SYSTEM_PROMPT = """
You are NutriBot, a nutrition assistant.

For calories questions, use get_calories.
For protein questions, use get_protein.

You MUST use MCP tools for nutrition values.
Never invent nutrition values.

All dataset values are per 100 grams.
If food is unavailable, clearly explain that.
"""


class NutritionAgent:

    def __init__(self):

        api_key = userdata.get("GROQ_API_KEY").strip()

        self.llm = Groq(api_key=api_key)

    async def ask(self, question):

        trace = []

        async with mcp_client.connect() as client:

            tools = await mcp_client.discover_tools(client)

            groq_tools = []

            for tool in tools:

                groq_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description or "",
                        "parameters": tool.input_schema
                    }
                })

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question}
            ]

            for _ in range(4):

                response = self.llm.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=groq_tools,
                    tool_choice="auto",
                    temperature=0
                )

                msg = response.choices[0].message

                if not msg.tool_calls:

                    return msg.content or "", trace

                messages.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {
                            "id": call.id,
                            "type": "function",
                            "function": {
                                "name": call.function.name,
                                "arguments": call.function.arguments
                            }
                        }
                        for call in msg.tool_calls
                    ]
                })

                for call in msg.tool_calls:

                    name = call.function.name

                    try:
                        args = json.loads(
                            call.function.arguments or "{}"
                        )

                        result, is_error = await mcp_client.call_tool(
                            client,
                            name,
                            args
                        )

                    except Exception as e:

                        result = str(e)
                        is_error = True

                    trace.append({
                        "tool": name,
                        "arguments": args if isinstance(args, dict) else {},
                        "result": result,
                        "is_error": is_error
                    })

                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": result
                    })

            return "Unable to complete the request.", trace

Overwriting nutrition_agent.py


In [41]:
import importlib
import mcp_client
import nutrition_agent

importlib.reload(mcp_client)
importlib.reload(nutrition_agent)

from nutrition_agent import NutritionAgent

agent = NutritionAgent()

print("NutriBot initialized successfully!")

NutriBot initialized successfully!


In [42]:
question = "How many calories are in 100g of chicken breast?"

answer, trace = await agent.ask(question)

print("User:", question)
print("NutriBot:", answer)

print("\nTool Trace:")

for item in trace:
    print(item)

User: How many calories are in 100g of chicken breast?
NutriBot: The calorie content of 100 g of chicken breast is **165 kcal**.

Tool Trace:
{'tool': 'get_calories', 'arguments': {'food_name': 'chicken breast'}, 'result': '165 kcal', 'is_error': False}


In [43]:
question = "How much protein is present in 100g of salmon?"

answer, trace = await agent.ask(question)

print("User:", question)
print("NutriBot:", answer)

print("\nTool Trace:")

for item in trace:
    print(item)

User: How much protein is present in 100g of salmon?
NutriBot: Salmon contains **20 g of protein per 100 g**.

Tool Trace:
{'tool': 'get_protein', 'arguments': {'food_name': 'salmon'}, 'result': '20 g protein', 'is_error': False}


In [44]:
question = "How many calories are in pizza?"

answer, trace = await agent.ask(question)

print("User:", question)
print("NutriBot:", answer)

print("\nTool Trace:")

for item in trace:
    print(item)

User: How many calories are in pizza?
NutriBot: I’m sorry, but “pizza” isn’t in the nutrition database I have access to. The available foods are:

- chicken breast  
- salmon  
- egg  
- broccoli  
- tofu  

If you have a specific type of pizza (e.g., pepperoni, margherita, or a particular brand) and can provide a name that matches the database, I can look up its calorie content. Otherwise, I can give you a general estimate based on typical values, but it won’t come from the official dataset. Let me know how you’d like to proceed!

Tool Trace:
{'tool': 'get_calories', 'arguments': {'food_name': 'pizza'}, 'result': 'Error executing tool get_calories: pizza not found. Available foods: chicken breast, salmon, egg, broccoli, tofu', 'is_error': True}


In [ ]:
import gradio as gr


async def chat_with_nutribot(message, history):

    answer, trace = await agent.ask(message)

    return answer


examples = [
    "How many calories are in 100g of chicken breast?",
    "How much protein does salmon contain?",
    "How many calories are in one egg per 100g?",
    "How much protein is present in tofu?",
    "What is the protein content of broccoli?"
]


demo = gr.ChatInterface(
    fn=chat_with_nutribot,
    title="NutriBot - Nutrition AI Assistant",
    description="Ask questions about calories and protein in foods.",
    examples=examples
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1476b385875c61355f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from importlib.metadata import version

packages = ["mcp", "groq", "pandas", "gradio"]

with open("requirements.txt", "w") as f:

    for package in packages:
        f.write(f"{package}=={version(package)}\n")

print(open("requirements.txt").read())

PackageNotFoundError: No package metadata was found for mcp

In [4]:
import importlib.metadata

try:
    print("MCP Version:", importlib.metadata.version("mcp"))
except importlib.metadata.PackageNotFoundError:
    print("MCP package is not installed!")


MCP Version: 2.3.0


In [3]:
%pip install --no-cache-dir mcp==2.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mcp]


In [5]:
import importlib.metadata

print("MCP Version:", importlib.metadata.version("mcp"))

MCP Version: 2.3.0


In [8]:
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

print("MCP installation and imports successful!")

MCP installation and imports successful!


In [9]:
from importlib.metadata import version

packages = ["mcp", "groq", "pandas", "gradio"]

with open("requirements.txt", "w") as f:

    for package in packages:
        f.write(f"{package}=={version(package)}\n")

print(open("requirements.txt").read())

PackageNotFoundError: No package metadata was found for groq

In [10]:
%pip install --no-cache-dir mcp==2.3.0 groq==1.7.0 pandas==2.2.3 gradio

In [11]:
from importlib.metadata import version, PackageNotFoundError

packages = ["mcp", "groq", "pandas", "gradio"]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

mcp: 2.3.0
groq: 1.7.0
pandas: 2.2.3
gradio: 6.31.0


In [12]:
from groq import Groq

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

import pandas as pd
import gradio as gr

print("All imports successful!")

All imports successful!


In [13]:
from importlib.metadata import version

packages = ["mcp", "groq", "pandas", "gradio"]

for package in packages:
    print(f"{package}: {version(package)}")

mcp: 2.3.0
groq: 1.7.0
pandas: 2.2.3
gradio: 6.31.0


In [14]:
import pandas as pd
from pathlib import Path

csv_path = Path("nutrition_data.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Dataset loaded successfully!")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())
else:
    print("nutrition_data.csv not found. Please upload your dataset.")

Dataset loaded successfully!
Shape: (5, 3)
Columns: ['food_name', 'calories_per_100g', 'protein_per_100g']


,food_name,calories_per_100g,protein_per_100g
0,chicken breast,165,31.0
1,salmon,208,20.0
2,egg,155,13.0
3,broccoli,34,2.8
4,tofu,76,8.0


In [15]:
from google.colab import files

uploaded = files.upload()

Saving nutrition_data.csv to nutrition_data (1).csv


In [16]:
from google.colab import files

uploaded = files.upload()

Saving mcp_client.py to mcp_client (1).py


In [17]:
from google.colab import files

uploaded = files.upload()

Saving mcp_server.py to mcp_server (1).py


In [18]:
from google.colab import files

uploaded = files.upload()

Saving nutrition_data.csv to nutrition_data (2).csv


In [19]:
%%writefile mcp_server.py

import logging
import sys
from pathlib import Path

import pandas as pd
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

# Send logs to stderr, not stdout
logging.basicConfig(
    stream=sys.stderr,
    level=logging.INFO
)

logger = logging.getLogger("nutribot")

# Dataset location
CSV_PATH = Path(__file__).parent / "nutrition_data.csv"

# Create MCP server
mcp = MCPServer("nutribot")


# Load nutrition dataset
def load_data():

    if not CSV_PATH.exists():
        raise ToolError("Nutrition dataset not found.")

    df = pd.read_csv(CSV_PATH)

    required_columns = [
        "food_name",
        "calories_per_100g",
        "protein_per_100g"
    ]

    for column in required_columns:
        if column not in df.columns:
            raise ToolError(f"Missing dataset column: {column}")

    df["food_name"] = (
        df["food_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return df


# Find food
def find_food(food_name):

    if not isinstance(food_name, str) or not food_name.strip():
        raise ToolError("Please provide a valid food name.")

    food_name = " ".join(food_name.strip().lower().split())

    df = load_data()

    result = df[df["food_name"] == food_name]

    if result.empty:

        available = ", ".join(df["food_name"].tolist())

        raise ToolError(
            f"{food_name} not found. Available foods: {available}"
        )

    return result.iloc[0]


# Tool 1: Get calories
@mcp.tool()
def get_calories(food_name: str) -> str:
    """Get calories per 100 grams of a food."""

    row = find_food(food_name)

    logger.info("Calories requested for %s", food_name)

    return f"{row['calories_per_100g']:g} kcal"


# Tool 2: Get protein
@mcp.tool()
def get_protein(food_name: str) -> str:
    """Get protein per 100 grams of a food."""

    row = find_food(food_name)

    logger.info("Protein requested for %s", food_name)

    return f"{row['protein_per_100g']:g} g protein"


# Start MCP server
if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting mcp_server.py


In [20]:
import py_compile

py_compile.compile("mcp_server.py", doraise=True)

print("MCP Server syntax is correct!")

MCP Server syntax is correct!


In [21]:
%%writefile mcp_client.py

import sys
from pathlib import Path
from contextlib import asynccontextmanager

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp_types import TextContent

SERVER_PATH = str(Path(__file__).parent / "mcp_server.py")

LOG_PATH = Path(__file__).parent / "mcp_server.log"


def server_params():

    return StdioServerParameters(
        command=sys.executable,
        args=[SERVER_PATH]
    )


@asynccontextmanager
async def connect():

    with open(LOG_PATH, "w", encoding="utf-8") as log_file:

        async with Client(
            stdio_client(
                server_params(),
                errlog=log_file
            )
        ) as client:

            yield client


# Discover available tools
async def discover_tools(client):

    result = await client.list_tools()

    return result.tools


# Execute selected tool
async def call_tool(client, name, arguments):

    result = await client.call_tool(name, arguments)

    output = "\n".join(
        item.text
        for item in result.content
        if isinstance(item, TextContent)
    )

    return output, bool(result.is_error)

Overwriting mcp_client.py


In [22]:
py_compile.compile("mcp_client.py", doraise=True)

print("MCP Client syntax is correct!")

MCP Client syntax is correct!


In [23]:
import importlib
import mcp_client

importlib.reload(mcp_client)

<module 'mcp_client' from '/content/mcp_client.py'>

In [24]:
async with mcp_client.connect() as client:

    print("MCP connection successful!")

    tools = await mcp_client.discover_tools(client)

    print("\nAvailable MCP Tools:")

    for tool in tools:
        print("-", tool.name)

MCP connection successful!

Available MCP Tools:
- get_calories
- get_protein


In [25]:
async with mcp_client.connect() as client:

    result, error = await mcp_client.call_tool(
        client,
        "get_calories",
        {"food_name": "chicken breast"}
    )

    print("Calories:", result)
    print("Error:", error)

Calories: 165 kcal
Error: False


In [26]:
async with mcp_client.connect() as client:

    result, error = await mcp_client.call_tool(
        client,
        "get_protein",
        {"food_name": "salmon"}
    )

    print("Protein:", result)
    print("Error:", error)

Protein: 20 g protein
Error: False


In [27]:
%%writefile nutrition_agent.py

import json
from groq import Groq
from google.colab import userdata

import mcp_client

MODEL = "openai/gpt-oss-20b"

SYSTEM_PROMPT = """
You are NutriBot, a friendly nutrition assistant.

Rules:
1. For calorie questions, use the get_calories tool.
2. For protein questions, use the get_protein tool.
3. Always use MCP tools to retrieve nutrition values.
4. Never guess or invent nutrition values.
5. All nutrition values are per 100 grams.
6. If a food is unavailable, explain that clearly.
7. Answer in simple and friendly language.
"""


class NutritionAgent:

    def __init__(self):

        api_key = userdata.get("GROQ_API_KEY")

        if not api_key:
            raise ValueError("GROQ_API_KEY is missing from Colab Secrets.")

        self.llm = Groq(api_key=api_key.strip())

        self.model = MODEL

    async def ask(self, question):

        trace = []

        async with mcp_client.connect() as client:

            # Discover MCP tools
            tools = await mcp_client.discover_tools(client)

            # Convert MCP tools into Groq function format
            groq_tools = []

            for tool in tools:

                groq_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description or "",
                        "parameters": tool.input_schema
                    }
                })

            messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": question
                }
            ]

            # Agent execution loop
            for _ in range(4):

                response = self.llm.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    tools=groq_tools,
                    tool_choice="auto",
                    temperature=0
                )

                msg = response.choices[0].message

                # If no tool is required, return final answer
                if not msg.tool_calls:

                    return msg.content or "", trace

                # Add assistant tool calls to conversation
                messages.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {
                            "id": call.id,
                            "type": "function",
                            "function": {
                                "name": call.function.name,
                                "arguments": call.function.arguments
                            }
                        }
                        for call in msg.tool_calls
                    ]
                })

                # Execute MCP tools
                for call in msg.tool_calls:

                    name = call.function.name

                    try:
                        args = json.loads(
                            call.function.arguments or "{}"
                        )

                        if not isinstance(args, dict):
                            raise ValueError("Tool arguments must be a JSON object.")

                        result, is_error = await mcp_client.call_tool(
                            client,
                            name,
                            args
                        )

                    except Exception as e:

                        args = {}
                        result = f"Tool execution failed: {e}"
                        is_error = True

                    trace.append({
                        "tool": name,
                        "arguments": args,
                        "result": result,
                        "is_error": is_error
                    })

                    # Send tool result back to Groq
                    messages.append({
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": result
                    })

            return "Sorry, I could not complete your request.", trace

Overwriting nutrition_agent.py


In [28]:
import py_compile

py_compile.compile(
    "nutrition_agent.py",
    doraise=True
)

print("Nutrition Agent syntax is correct!")

Nutrition Agent syntax is correct!


In [29]:
import importlib
import mcp_client
import nutrition_agent

importlib.reload(mcp_client)
importlib.reload(nutrition_agent)

from nutrition_agent import NutritionAgent

agent = NutritionAgent()

print("NutriBot initialized successfully!")

NutriBot initialized successfully!


In [30]:
question = "How many calories are in 100g of chicken breast?"

answer, trace = await agent.ask(question)

print("USER:", question)
print("\nNUTRIBOT:", answer)

print("\n--- MCP TOOL EXECUTION ---")

for item in trace:
    print("Tool:", item["tool"])
    print("Arguments:", item["arguments"])
    print("Result:", item["result"])
    print("Error:", item["is_error"])

USER: How many calories are in 100g of chicken breast?

NUTRIBOT: Chicken breast has about **165 calories per 100 g**.

--- MCP TOOL EXECUTION ---
Tool: get_calories
Arguments: {'food_name': 'chicken breast'}
Result: 165 kcal
Error: False


In [31]:
question = "How much protein is present in 100g of salmon?"

answer, trace = await agent.ask(question)

print("USER:", question)
print("\nNUTRIBOT:", answer)

print("\n--- MCP TOOL EXECUTION ---")

for item in trace:
    print(item)

USER: How much protein is present in 100g of salmon?

NUTRIBOT: Sure thing! Salmon has about **20 g of protein per 100 g**. Enjoy your meal!

--- MCP TOOL EXECUTION ---
{'tool': 'get_protein', 'arguments': {'food_name': 'salmon'}, 'result': '20 g protein', 'is_error': False}


In [32]:
question = "How many calories are in pizza?"

answer, trace = await agent.ask(question)

print("USER:", question)
print("\nNUTRIBOT:", answer)

print("\n--- MCP TOOL EXECUTION ---")

for item in trace:
    print(item)

USER: How many calories are in pizza?

NUTRIBOT: I’m sorry, but I don’t have data for pizza in my database. I can give you calorie information for foods like chicken breast, salmon, egg, broccoli, or tofu if that helps!

--- MCP TOOL EXECUTION ---
{'tool': 'get_calories', 'arguments': {'food_name': 'pizza'}, 'result': 'Error executing tool get_calories: pizza not found. Available foods: chicken breast, salmon, egg, broccoli, tofu', 'is_error': True}
